In [1]:
import os
import glob
import os
from scipy.io import mmwrite
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml
import time
import gget
import psutil
from scipy.stats import zscore

import rmm
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator
import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

sc.settings.verbosity = 3

/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

In [3]:
%%time
fpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/processed/sample_1_adata.h5ad"
adata = sc.read_h5ad(fpath)
adata

CPU times: user 4.59 s, sys: 27.6 s, total: 32.2 s
Wall time: 43.7 s


AnnData object with n_obs × n_vars = 1125041 × 52164
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'X_umap_X_scANVI', 'X_umap_raw_data', 'X_umap_scVI', '_scvi_manager_uuid', '_scvi_uuid', 'basename_colors', 'basename_palette

In [4]:
rsc.get.anndata_to_GPU(adata) # move to GPU
rsc.pp.filter_genes(adata, min_counts=500)
adata

filtered out 23626 genes that are detected in less than 500 counts


AnnData object with n_obs × n_vars = 1125041 × 28538
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'X_umap_X_scANVI', 'X_umap_raw_data', 'X_umap_scVI', '_scvi_manager_uuid', '_scvi_uuid', 'basename_colors', 'basename_palette

In [5]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/basename_ref.h5ad"

aggdata = sc.get.aggregate(
    adata,
    by='basename',
    func='sum',
    layer='counts',
)

aggdata.X = aggdata.layers['sum']
del aggdata.layers['sum']
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()


aggdata.shape=(8, 28538)
CPU times: user 5.37 s, sys: 4.91 s, total: 10.3 s
Wall time: 10.4 s


,basename
endothelial_cells,endothelial_cells
fibroblasts,fibroblasts
hematopoietic_progenitors,hematopoietic_progenitors
hsc,hsc
innate_lymphoid_cells,innate_lymphoid_cells


In [6]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/cell_type_ref.h5ad"

aggdata = sc.get.aggregate(
    adata,
    by='cell_type',
    func='sum',
    layer='counts',
)

aggdata.X = aggdata.layers['sum']
del aggdata.layers['sum']
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()


aggdata.shape=(188, 28538)
CPU times: user 6.17 s, sys: 5.68 s, total: 11.8 s
Wall time: 12.1 s


,cell_type
B-1 B cell,B-1 B cell
B-1a B cell,B-1a B cell
B-1b B cell,B-1b B cell
B-2 B cell,B-2 B cell
CD1c-positive myeloid dendritic cell,CD1c-positive myeloid dendritic cell


In [7]:
sorted(adata.obs['cell_type'].unique())

['B-1 B cell',
 'B-1a B cell',
 'B-1b B cell',
 'B-2 B cell',
 'CD141-positive myeloid dendritic cell',
 'CD16-negative, CD56-bright natural killer cell, human',
 'CD16-positive, CD56-dim natural killer cell, human',
 'CD1c-positive myeloid dendritic cell',
 'CD34-positive, CD38-negative hematopoietic stem cell',
 'CD34-positive, CD56-positive, CD117-positive common innate lymphoid precursor, human',
 'CD4-positive helper T cell',
 'CD4-positive, CD25-positive, alpha-beta regulatory T cell',
 'CD4-positive, alpha-beta cytotoxic T cell',
 'CD4-positive, alpha-beta memory T cell',
 'CD4-positive, alpha-beta thymocyte',
 'CD8-alpha alpha positive, gamma-delta intraepithelial T cell',
 'CD8-alpha-alpha-positive, alpha-beta intraepithelial T cell',
 'CD8-alpha-beta-positive, alpha-beta intraepithelial T cell',
 'CD8-positive, alpha-beta cytokine secreting effector T cell',
 'CD8-positive, alpha-beta cytotoxic T cell',
 'CD8-positive, alpha-beta memory T cell',
 'CD8-positive, alpha-beta thy

# hand-selected-cell-types

In [8]:
%%time
cell_types = [
    'alveolar type 2 fibroblast cell',
    'cord blood hematopoietic stem cell',
    'dermis microvascular lymphatic vessel endothelial cell',
    'embryonic fibroblast',
    'endothelial cell of pericentral hepatic sinusoid',
    'endothelial cell of periportal hepatic sinusoid',
    'endothelial cell of placenta',
    'endothelial cell of venule',
    'erythroid progenitor cell',
    'fibro/adipogenic progenitor cell',
    'fibroblast of breast',
    'fibroblast of connective tissue of glandular part of prostate',
    'fibroblast of lung',
    'kidney interstitial fibroblast',
    'megakaryocyte',
    'mesothelial cell',
    'primitive red blood cell',
    'prostate gland microvascular endothelial cell',
    'skin fibroblast',
    'smooth muscle cell of the pulmonary artery'
]

bdata = adata[adata.obs['cell_type'].isin(cell_types), :].copy()
print(bdata.obs['cell_type'].value_counts().to_string())

cell_type
erythroid progenitor cell                                        23229
skin fibroblast                                                  17272
mesothelial cell                                                 17141
cord blood hematopoietic stem cell                                9056
fibroblast of lung                                                5058
megakaryocyte                                                     3578
fibro/adipogenic progenitor cell                                  2509
endothelial cell of placenta                                      2390
endothelial cell of venule                                        2295
endothelial cell of pericentral hepatic sinusoid                  2124
alveolar type 2 fibroblast cell                                   1954
embryonic fibroblast                                              1709
primitive red blood cell                                          1523
kidney interstitial fibroblast                                    1

In [9]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/hand_select.h5ad"

aggdata = sc.get.aggregate(
    bdata,
    by='cell_type',
    func='sum',
    layer='counts',
)

aggdata.X = aggdata.layers['sum']
del aggdata.layers['sum']
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()

aggdata.shape=(20, 28538)
CPU times: user 483 ms, sys: 432 ms, total: 915 ms
Wall time: 963 ms


,cell_type
alveolar type 2 fibroblast cell,alveolar type 2 fibroblast cell
cord blood hematopoietic stem cell,cord blood hematopoietic stem cell
dermis microvascular lymphatic vessel endothelial cell,dermis microvascular lymphatic vessel endothel...
embryonic fibroblast,embryonic fibroblast
endothelial cell of pericentral hepatic sinusoid,endothelial cell of pericentral hepatic sinusoid
